# 02 · Layers: stacking inlists

Each inlist the setup writes is built for one **target**: `binary_controls`, `binary_job`, `star1_controls`, `star1_job`, …, and for stars built in steps, `star1_formation_step0_controls` and so on. For every target, values come from four **layers**, applied in a fixed, named order:

```
1. mesa_inlists      MESA's own defaults
2. posydon_inlists   POSYDON's physics, picked by the scenario
3. user_inlists      your own files
4. generated         keys the setup writes itself
```

A later layer overrides an earlier one **key by key**: it changes only the keys it sets. This is `posydon/grids/setup/layers.py`. Before the rewrite, which file won depended on the order of the keys in the ini (tracker S12); now it depends only on the layer.

In [1]:
from pathlib import Path
from nb_env import env, write, fresh
ENV = env()

from posydon.grids.setup.layers import (
    LAYER_ORDER, LayerStack, drop_wiring_keys, expand_every_step,
    fix_num_x_ctrls, formation_step_count, pick_whole_file, section_for)
LAYER_ORDER

('mesa_inlists', 'posydon_inlists', 'user_inlists', 'generated')

## A toy stack

In [2]:
stack = LayerStack("star1_controls")
stack.add("mesa_inlists", {"mesh_delta_coeff": "1d0", "alpha_semiconvection": "0d0", "max_age": "1d36"})
stack.add("posydon_inlists", {"alpha_semiconvection": "0.1d0", "max_age": "1.4e10"}, source="POSYDON")
stack.add("user_inlists", {"mesh_delta_coeff": "0.8d0"}, source="my_inlist1")
stack.merged()

{'mesh_delta_coeff': '0.8d0',
 'alpha_semiconvection': '0.1d0',
 'max_age': '1.4e10'}

`explain(key)` gives every layer that set a key, lowest first; the last one wins. This is what `posydon-grid explain` prints.

In [3]:
for key in ["mesh_delta_coeff", "alpha_semiconvection", "max_age"]:
    print(key, [(e["layer"], e["source"], e["value"]) for e in stack.explain(key)])

mesh_delta_coeff [('mesa_inlists', None, '1d0'), ('user_inlists', 'my_inlist1', '0.8d0')]
alpha_semiconvection [('mesa_inlists', None, '0d0'), ('posydon_inlists', 'POSYDON', '0.1d0')]
max_age [('mesa_inlists', None, '1d36'), ('posydon_inlists', 'POSYDON', '1.4e10')]


### The order of `add` calls does not matter between layers

Here the user layer is added first, then MESA's defaults: the result is the same. Within one layer, files are applied in the order they were added (that is how a rerun's overlay goes right after the file it changes, notebook 04).

In [4]:
reversed_stack = LayerStack("star1_controls")
reversed_stack.add("user_inlists", {"mesh_delta_coeff": "0.8d0"})
reversed_stack.add("posydon_inlists", {"alpha_semiconvection": "0.1d0", "max_age": "1.4e10"})
reversed_stack.add("mesa_inlists", {"mesh_delta_coeff": "1d0", "alpha_semiconvection": "0d0", "max_age": "1d36"})
reversed_stack.merged() == stack.merged()

True

A key keeps the position where it **first** appears (so MESA's default order is kept, as in today's files), with the value of the last layer that set it:

In [5]:
list(reversed_stack.merged())

['mesh_delta_coeff', 'alpha_semiconvection', 'max_age']

Unknown layers are refused:

In [6]:
try:
    stack.add("generated_typo", {})
except ValueError as error:
    print("ValueError:", error)

ValueError: unknown layer 'generated_typo'; layers are mesa_inlists, posydon_inlists, user_inlists, generated


## Targets and sections

A target says which section of an input file it reads: binary targets read their own section, star targets read `&controls` or `&star_job`.

In [7]:
for target in ["binary_controls", "binary_job", "star1_controls", "star2_job", "star1_formation_step1_controls"]:
    print(f"{target:32} <- &{section_for(target)}")

binary_controls                  <- &binary_controls
binary_job                       <- &binary_job
star1_controls                   <- &controls
star2_job                        <- &star_job
star1_formation_step1_controls   <- &controls


## A real stack: star 1 of a CO-HeMS grid

`add_file` reads the target's section from a file (or a whole `.defaults` file). This is what the setup does for `star1_controls` of a CO-HeMS grid, plus a user file on top. `clean` is the clean-up the setup applies to every star-controls input file (explained at the end of this notebook).

In [8]:
defaults = ENV.inlists / "r11701" / "mesa" / "default" / "defaults"
my_inlist = write(fresh("02_layers") / "my_inlist1", """
    &controls
        mesh_delta_coeff = 0.8d0
        alpha_semiconvection = 0.2d0
    /
    """)

def clean(values):
    return fix_num_x_ctrls(drop_wiring_keys(values))

star1 = LayerStack("star1_controls")
star1.add_file("mesa_inlists", defaults / "star" / "controls.defaults", transform=clean)
star1.add_file("posydon_inlists", ENV.release / "common" / "binary" / "inlist1", transform=clean)
star1.add_file("posydon_inlists", ENV.release / "CO-HeMS" / "binary" / "inlist1", transform=clean)
star1.add_file("user_inlists", my_inlist, transform=clean)

def where(source):
    return source.split("/r11701/")[-1] if "/r11701/" in source else Path(source).name

merged = star1.merged()
print(len(merged), "keys in the merged star1_controls")
for key in ["mesh_delta_coeff", "alpha_semiconvection"]:
    print(key)
    for e in star1.explain(key):
        print(f"   {e['value']:>10}  [{e['layer']}] {where(e['source'])}")

1657 keys in the merged star1_controls
mesh_delta_coeff
        1.0d0  [mesa_inlists] mesa/default/defaults/star/controls.defaults
        1.0d0  [posydon_inlists] releases/v2.0/common/binary/inlist1
        0.8d0  [user_inlists] my_inlist1
alpha_semiconvection
            0  [mesa_inlists] mesa/default/defaults/star/controls.defaults
        0.1d0  [posydon_inlists] releases/v2.0/common/binary/inlist1
        0.2d0  [user_inlists] my_inlist1


How many keys does each layer decide in the end?

In [9]:
from collections import Counter
winner = Counter(star1.explain(key)[-1]["layer"] for key in merged)
winner

Counter({'mesa_inlists': 1525, 'posydon_inlists': 130, 'user_inlists': 2})

And which keys does the grid type's own file (`CO-HeMS/binary/inlist1`) change on top of the common one?

In [10]:
by_source = {}
for key in merged:
    for e in star1.explain(key):
        if "CO-HeMS" in (e["source"] or ""):
            by_source[key] = e["value"]
by_source

{'x_ctrl(1)': '1.0d0'}

## Whole files are not merged

Column lists and Fortran sources are not merged key by key: the highest layer that has one wins outright.

In [11]:
pick_whole_file({"mesa_inlists": "mesa/history_columns.list",
                 "posydon_inlists": "common/history_columns.list",
                 "user_inlists": None})

('posydon_inlists', 'common/history_columns.list')

In [12]:
pick_whole_file({"mesa_inlists": "mesa/history_columns.list",
                 "posydon_inlists": "common/history_columns.list",
                 "user_inlists": "my_history_columns.list"})

('user_inlists', 'my_history_columns.list')

## Formation steps

Stars built in steps (CO-HeMS, HeMS-HMS, single_HeMS) have one target per step. A file meant for **every** step goes into the `<star>_formation_every_step_<part>` target; `expand_every_step` puts it in front of each step's own files. `formation_step_count` finds the number of steps across all layers.

In [13]:
layer = {"star1_formation_every_step_controls": ["common/binary/inlist1"],
         "star1_formation_step0_controls": ["CO-HeMS/star1_formation/inlist_step1"],
         "star1_formation_step1_controls": ["CO-HeMS/star1_formation/inlist_step2"]}
n = formation_step_count("star1", [layer])
print(n, "steps")
expand_every_step(layer, "star1", n)

2 steps


{'star1_formation_step0_controls': ['common/binary/inlist1',
  'CO-HeMS/star1_formation/inlist_step1'],
 'star1_formation_step1_controls': ['common/binary/inlist1',
  'CO-HeMS/star1_formation/inlist_step2']}

## Two clean-ups applied to every input file

Today's setup applies these to each input file before stacking, and the rewrite keeps them:

- `drop_wiring_keys`: input files may point MESA at further inlists (`read_extra_*`, `*inlist*` keys). The setup writes its own wiring, so these are dropped;
- `fix_num_x_ctrls`: MESA's defaults write `x_ctrl(1:num_x_ctrls) = 0d0`, which MESA cannot read back; only element 1 gets the default (tracker C2).

In [14]:
drop_wiring_keys({"read_extra_controls_inlist1": ".true.", "extra_controls_inlist1_name": "'x'", "mesh_delta_coeff": "1d0"})

{'mesh_delta_coeff': '1d0'}

In [15]:
fix_num_x_ctrls({"x_ctrl(1:num_x_ctrls)": "0d0", "x_logical_ctrl(1:num_x_ctrls)": ".false.", "max_age": "1d36"})

{'x_ctrl(1:1)': '0d0', 'x_logical_ctrl(1:1)': '.false.', 'max_age': '1d36'}

## Try it

- Add a second user file to `star1` that sets `mesh_delta_coeff` again; which one wins?
- Build the `star1_formation_step1_controls` stack of CO-HeMS by hand (MESA defaults, `common/binary/inlist1`, `CO-HeMS/star1_formation/inlist_step2`) and compare it with what notebook 05 builds.